In [1]:
import numpy as np
import tensorflow as tf

In [2]:
# movies × users
Y = np.array([
    [5, 4, 0, 0],
    [3, 0, 0, 0],
    [4, 0, 0, 0],
    [3, 0, 0, 0],
    [3, 0, 0, 0],
])

R = (Y != 0).astype(int)

num_movies, num_users = Y.shape
num_features = 3

In [3]:
tf.random.set_seed(0)

X = tf.Variable(tf.random.normal((num_movies, num_features), dtype=tf.float64))
W = tf.Variable(tf.random.normal((num_users, num_features), dtype=tf.float64))
b = tf.Variable(tf.zeros((1, num_users), dtype=tf.float64))

In [4]:
def compute_cost(X, W, b, Y, R, lambda_):
    pred = tf.matmul(X, tf.transpose(W)) + b
    error = (pred - Y) * R

    cost = 0.5 * tf.reduce_sum(error**2)
    reg = (lambda_/2) * (tf.reduce_sum(W**2) + tf.reduce_sum(X**2))

    return cost + reg

In [5]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.1)

lambda_ = 1
iterations = 200

for i in range(iterations):

    with tf.GradientTape() as tape:
        cost = compute_cost(X, W, b, Y, R, lambda_)

    grads = tape.gradient(cost, [X, W, b])
    optimizer.apply_gradients(zip(grads, [X, W, b]))

    if i % 20 == 0:
        print(f"Iteration {i}, Cost: {cost.numpy():.2f}")

Iteration 0, Cost: 86.67
Iteration 20, Cost: 8.16
Iteration 40, Cost: 3.62
Iteration 60, Cost: 2.11
Iteration 80, Cost: 1.56
Iteration 100, Cost: 1.35
Iteration 120, Cost: 1.29
Iteration 140, Cost: 1.29
Iteration 160, Cost: 1.29
Iteration 180, Cost: 1.29


In [6]:
predictions = tf.matmul(X, tf.transpose(W)) + b
predictions = predictions.numpy()

print("Predicted Ratings:\n", predictions)

Predicted Ratings:
 [[ 4.21755191e+00  3.99988254e+00  3.35569461e-05 -1.31172346e-05]
 [ 3.33548113e+00  3.99967794e+00 -1.43921997e-05  5.62540071e-06]
 [ 3.77645025e+00  3.99978022e+00  9.58208161e-06 -3.74391353e-06]
 [ 3.33555079e+00  3.99967795e+00 -1.43897745e-05  5.62411119e-06]
 [ 3.33548086e+00  3.99967792e+00 -1.43956532e-05  5.62695000e-06]]


In [8]:
user_id = 1

user_ratings = predictions[:, user_id]

sorted_idx = np.argsort(user_ratings)[::-1]

print("\nTop recommendations:")
for i in sorted_idx:
    if R[i, user_id] == 0:
        print(f"Movie {i} → predicted rating {user_ratings[i]:.2f}")


Top recommendations:
Movie 2 → predicted rating 4.00
Movie 3 → predicted rating 4.00
Movie 1 → predicted rating 4.00
Movie 4 → predicted rating 4.00


In [9]:
!wget https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip ml-latest-small.zip

--2026-04-01 16:47:32--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip’

ml-latest-small.zip 100%[===================>] 955.28K  --.-KB/s    in 0.1s    

2026-04-01 16:47:32 (6.45 MB/s) - ‘ml-latest-small.zip’ saved [978202/978202]

Archive:  ml-latest-small.zip
   creating: ml-latest-small/
  inflating: ml-latest-small/links.csv  
  inflating: ml-latest-small/tags.csv  
  inflating: ml-latest-small/ratings.csv  
  inflating: ml-latest-small/README.txt  
  inflating: ml-latest-small/movies.csv  


In [10]:
import pandas as pd
import numpy as np

ratings = pd.read_csv("ml-latest-small/ratings.csv")
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [11]:
# Create pivot table (movies × users)
Y_df = ratings.pivot(index='movieId', columns='userId', values='rating')

# Fill missing with 0
Y = Y_df.fillna(0).values

# Indicator matrix
R = (Y != 0).astype(int)

print("Y shape:", Y.shape)
print("R shape:", R.shape)

Y shape: (9724, 610)
R shape: (9724, 610)


In [12]:
# Create pivot table (movies × users)
Y_df = ratings.pivot(index='movieId', columns='userId', values='rating')

# Fill missing with 0
Y = Y_df.fillna(0).values

# Indicator matrix
R = (Y != 0).astype(int)

print("Y shape:", Y.shape)
print("R shape:", R.shape)

Y shape: (9724, 610)
R shape: (9724, 610)


In [13]:
Y = Y[:500, :200]
R = R[:500, :200]

In [14]:
p = tf.matmul(X, tf.transpose(W)) + b
p = p.numpy()

In [18]:
print("p:", p.shape)
print("Y:", Y.shape)
print("R:", R.shape)

p: (5, 4)
Y: (500, 200)
R: (500, 200)


In [19]:
Ymean = np.sum(Y, axis=1) / (np.sum(R, axis=1) + 1e-8)
Ymean = Ymean.reshape(-1, 1)

In [21]:
# recompute p using current Y size
p = tf.matmul(X, tf.transpose(W)) + b
p = p.numpy()

print(p.shape)  # must match Y.shape

(5, 4)


In [22]:
Ymean = np.sum(Y, axis=1) / (np.sum(R, axis=1) + 1e-8)
Ymean = Ymean.reshape(-1, 1)

In [23]:
print(Ymean.shape)   # should be (5,1)

(500, 1)


In [26]:
# small dataset
Y = np.array([
    [5, 4, 0, 0],
    [3, 0, 0, 0],
    [4, 0, 0, 0],
    [3, 0, 0, 0],
    [3, 0, 0, 0],
])

R = (Y != 0).astype(int)

num_movies, num_users = Y.shape
num_features = 3

In [27]:
import tensorflow as tf

X = tf.Variable(tf.random.normal((num_movies, num_features), dtype=tf.float64))
W = tf.Variable(tf.random.normal((num_users, num_features), dtype=tf.float64))
b = tf.Variable(tf.zeros((1, num_users), dtype=tf.float64))

In [28]:
p = tf.matmul(X, tf.transpose(W)) + b
p = p.numpy()

print(p.shape)  # must be (5,4)

(5, 4)


In [29]:
Ymean = np.sum(Y, axis=1) / (np.sum(R, axis=1) + 1e-8)
Ymean = Ymean.reshape(-1, 1)

print(Ymean.shape)  # must be (5,1)

(5, 1)


In [30]:
pm = p + Ymean

In [31]:
user_id = 1   # pick a user with missing entries

my_predictions = pm[:, user_id]
sorted_idx = np.argsort(my_predictions)[::-1]

print("\nTop recommendations:\n")

for i in sorted_idx:
    if R[i, user_id] == 0:
        print(f"Movie {i} → Predicted rating {my_predictions[i]:.2f}")


Top recommendations:

Movie 2 → Predicted rating 3.55
Movie 1 → Predicted rating 3.38
Movie 4 → Predicted rating 3.02
Movie 3 → Predicted rating 1.96


In [32]:
print("\nOriginal vs Predicted:\n")

for i in range(len(my_predictions)):
    if R[i, user_id] == 1:
        print(f"Movie {i} | Actual: {Y[i, user_id]} | Predicted: {my_predictions[i]:.2f}")


Original vs Predicted:

Movie 0 | Actual: 4 | Predicted: 3.22


In [33]:
error = (pm - Y) * R
rmse = np.sqrt(np.sum(error**2) / np.sum(R))

print("RMSE:", rmse)

RMSE: 1.1711028283404836


In [34]:
movie_names = [
    "Toy Story (1995)",
    "GoldenEye (1995)",
    "Four Rooms (1995)",
    "Get Shorty (1995)",
    "Copycat (1995)"
]

In [35]:
print(f"\nTop recommendations for user {user_id}:\n")

rank = 1
for i in sorted_idx:
    if R[i, user_id] == 0:
        print(f"{rank:2d}. {movie_names[i]:<20} | Predicted rating: {my_predictions[i]:.2f}")
        rank += 1
    if rank > 10:
        break


Top recommendations for user 1:

 1. Four Rooms (1995)    | Predicted rating: 3.55
 2. GoldenEye (1995)     | Predicted rating: 3.38
 3. Copycat (1995)       | Predicted rating: 3.02
 4. Get Shorty (1995)    | Predicted rating: 1.96


In [36]:
print("\nModel evaluation:\n")

for i in range(len(my_predictions)):
    if R[i, user_id] == 1:
        print(f"{movie_names[i]:<20} | Actual: {Y[i, user_id]:.1f} | Predicted: {my_predictions[i]:.2f}")


Model evaluation:

Toy Story (1995)     | Actual: 4.0 | Predicted: 3.22


In [37]:
movie_names = [
    "Toy Story (1995)",
    "GoldenEye (1995)",
    "Four Rooms (1995)",
    "Get Shorty (1995)",
    "Copycat (1995)",
    "Shanghai Triad (1995)",
    "Twelve Monkeys (1995)",
    "Babe (1995)",
    "Dead Man Walking (1995)",
    "Richard III (1995)",
    "Seven (1995)",
    "Usual Suspects (1995)",
    "Mighty Aphrodite (1995)",
    "Postino, Il (1994)",
    "Mr. Holland's Opus (1995)",
    "French Twist (1995)",
    "From Dusk Till Dawn (1996)",
    "White Balloon (1995)",
    "Antonia's Line (1995)",
    "Angels and Insects (1995)",
    "Braveheart (1995)",
    "Taxi Driver (1976)",
    "Rumble in the Bronx (1995)",
    "Apollo 13 (1995)",
    "Batman Forever (1995)",
    "Beauty of the Day (1967)",
    "Crimson Tide (1995)",
    "Desperado (1995)",
    "Doom Generation (1995)",
    "Free Willy 2 (1995)",
    "Mad Love (1995)",
    "Nadja (1994)",
    "Net, The (1995)",
    "Strange Days (1995)",
    "To Wong Foo (1995)",
    "Billy Madison (1995)",
    "Clerks (1994)",
    "Disclosure (1994)",
    "Dolores Claiborne (1995)",
    "Eat Drink Man Woman (1994)",
    "Exotica (1994)",
    "Ed Wood (1994)",
    "Hoop Dreams (1994)",
    "I.Q. (1994)",
    "Interview with the Vampire (1994)",
    "Legends of the Fall (1994)",
    "Little Women (1994)",
    "Miracle on 34th Street (1994)",
    "Natural Born Killers (1994)",
    "Outbreak (1995)"
]

In [38]:
len(movie_names) == Y.shape[0]

False

In [39]:
Y = Y[:50, :]
R = R[:50, :]

In [40]:
print(f"\nTop recommendations for user {user_id}:\n")

rank = 1
for i in sorted_idx:
    if R[i, user_id] == 0:
        print(f"{rank:2d}. {movie_names[i]:<30} | Predicted: {my_predictions[i]:.2f}")
        rank += 1
    if rank > 10:
        break


Top recommendations for user 1:

 1. Four Rooms (1995)              | Predicted: 3.55
 2. GoldenEye (1995)               | Predicted: 3.38
 3. Copycat (1995)                 | Predicted: 3.02
 4. Get Shorty (1995)              | Predicted: 1.96
